<a href="https://colab.research.google.com/github/09-ashish/DSA-CaseStudies/blob/main/Intermediate_Assessment_3.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [21]:
import pandas as pd
import numpy as np
from sklearn.ensemble import GradientBoostingClassifier, RandomForestClassifier
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import classification_report, f1_score
from sklearn.model_selection import GridSearchCV, train_test_split


In [4]:
df=pd.read_csv("/content/train_LZdllcl.csv")
df.head()


,employee_id,department,region,education,gender,recruitment_channel,no_of_trainings,age,previous_year_rating,length_of_service,KPIs_met >80%,awards_won?,avg_training_score,is_promoted
0,65438,Sales & Marketing,region_7,Master's & above,f,sourcing,1,35,5.0,8,1,0,49,0
1,65141,Operations,region_22,Bachelor's,m,other,1,30,5.0,4,0,0,60,0
2,7513,Sales & Marketing,region_19,Bachelor's,m,sourcing,1,34,3.0,7,0,0,50,0
3,2542,Sales & Marketing,region_23,Bachelor's,m,other,2,39,1.0,10,0,0,50,0
4,48945,Technology,region_26,Bachelor's,m,other,1,45,3.0,2,0,0,73,0


In [5]:
df.isnull().sum()

,0
employee_id,0
department,0
region,0
education,2409
gender,0
recruitment_channel,0
no_of_trainings,0
age,0
previous_year_rating,4124
length_of_service,0


In [22]:
df['previous_year_rating']=df['previous_year_rating'].fillna(df['previous_year_rating'].median())
df['education']=df['education'].fillna('Unknown')
df['kpi_eligible'] = df['KPIs_met >80%']
cat_cols=['department','region','education','gender','recruitment_channel']
df_encoded=pd.get_dummies(df,cat_cols,drop_first=True,dtype=int)
X=df_encoded.drop(columns=['employee_id','is_promoted'])
y=df_encoded['is_promoted']

In [23]:
X_train,X_test,y_train,y_test = train_test_split(X,y,test_size=0.2,random_state=42,stratify=y)




In [24]:
lr_model = LogisticRegression(max_iter=1000,class_weight='balanced',random_state=42)
lr_model.fit(X_train,y_train)

y_pred_lr = lr_model.predict(X_test)
f1_lr = f1_score(y_test,y_pred_lr,average='binary')

print(f"Logistic regression F1  Score: {f1_lr:.4f}")
print(classification_report(y_test,y_pred_lr))

Logistic regression F1  Score: 0.3749
              precision    recall  f1-score   support

           0       0.98      0.76      0.86     10028
           1       0.24      0.82      0.37       934

    accuracy                           0.77     10962
   macro avg       0.61      0.79      0.62     10962
weighted avg       0.92      0.77      0.82     10962



/usr/local/lib/python3.12/dist-packages/sklearn/linear_model/_logistic.py:465: ConvergenceWarning: lbfgs failed to converge (status=1):
STOP: TOTAL NO. OF ITERATIONS REACHED LIMIT.

Increase the number of iterations (max_iter) or scale the data as shown in:
    https://scikit-learn.org/stable/modules/preprocessing.html
Please also refer to the documentation for alternative solver options:
    https://scikit-learn.org/stable/modules/linear_model.html#logistic-regression
  n_iter_i = _check_optimize_result(


In [25]:
rf_model = RandomForestClassifier(n_estimators=100,class_weight='balanced',random_state=42)
rf_model.fit(X_train,y_train)

y_pred_rf = rf_model.predict(X_test)
f1_rf = f1_score(y_test,y_pred_rf,average='binary')

print(f"random forest F1  Score: {f1_rf:.4f}")
print(classification_report(y_test,y_pred_rf))

random forest F1  Score: 0.4376
              precision    recall  f1-score   support

           0       0.94      0.99      0.97     10028
           1       0.83      0.30      0.44       934

    accuracy                           0.94     10962
   macro avg       0.89      0.65      0.70     10962
weighted avg       0.93      0.94      0.92     10962



In [27]:
gb_model = GradientBoostingClassifier(n_estimators=100,random_state=42)
gb_model.fit(X_train,y_train)

y_pred_gb = gb_model.predict(X_test)
f1_gb = f1_score(y_test,y_pred_gb,average='binary')

print(f"random forest F1  Score: {f1_gb:.4f}")
print(classification_report(y_test,y_pred_gb))

random forest F1  Score: 0.4027
              precision    recall  f1-score   support

           0       0.94      1.00      0.97     10028
           1       0.94      0.26      0.40       934

    accuracy                           0.94     10962
   macro avg       0.94      0.63      0.68     10962
weighted avg       0.94      0.94      0.92     10962



In [28]:
from xgboost import XGBClassifier

scale_pos_weight = (len(y_train) - sum(y_train)) / sum(y_train)

xgb_model = XGBClassifier(
    n_estimators=100,
    learning_rate=0.1,
    scale_pos_weight=scale_pos_weight,
    random_state=42,
)
xgb_model.fit(X_train, y_train)


y_pred_xgb = xgb_model.predict(X_test)
f1_xgb = f1_score(y_test, y_pred_xgb, average='binary')

print(f"=== XGBoost F1 Score (Promoted): {f1_xgb:.4f} ===\n")
print(classification_report(y_test, y_pred_xgb))

=== XGBoost F1 Score (Promoted): 0.3846 ===

              precision    recall  f1-score   support

           0       0.99      0.75      0.85     10028
           1       0.25      0.88      0.38       934

    accuracy                           0.76     10962
   macro avg       0.62      0.81      0.62     10962
weighted avg       0.92      0.76      0.81     10962



In [29]:
param_grid={
    'n_estimators':[100,200],
    'max_depth':[15,25,None],
    'min_samples_split':[2,5],
    'min_samples_leaf':[1,2],
    'class_weight':['balanced','balanced_subsample'],
}

rf_grid = GridSearchCV(
    estimator=RandomForestClassifier(random_state=42),
    param_grid=param_grid,
    scoring='f1',
    cv=3,
    n_jobs=-1,
    verbose=1,
)

rf_grid.fit(X_train, y_train)

best_rf = rf_grid.best_estimator_
print("\nBest Parameters found:", rf_grid.best_params_)

y_pred_best = best_rf.predict(X_test)
f1_best = f1_score(y_test, y_pred_best, average='binary')

print(f"\n=== Tuned Random Forest F1 Score: {f1_best:.4f} ===\n")
print(classification_report(y_test, y_pred_best))


Fitting 3 folds for each of 48 candidates, totalling 144 fits

Best Parameters found: {'class_weight': 'balanced_subsample', 'max_depth': None, 'min_samples_leaf': 2, 'min_samples_split': 2, 'n_estimators': 200}

=== Tuned Random Forest F1 Score: 0.4738 ===

              precision    recall  f1-score   support

           0       0.95      0.95      0.95     10028
           1       0.48      0.47      0.47       934

    accuracy                           0.91     10962
   macro avg       0.71      0.71      0.71     10962
weighted avg       0.91      0.91      0.91     10962



In [30]:
import pandas as pd

test_df = pd.read_csv('/content/test_2umaH9m.csv')
sample_sub = pd.read_csv(
    '/content/sample_submission_M0L0uXE.csv'
)

test_df['previous_year_rating'] = test_df['previous_year_rating'].fillna(
    df['previous_year_rating'].median()
)
test_df['education'] = test_df['education'].fillna('Unknown')
test_df['kpi_eligible'] = test_df['KPIs_met >80%']

cat_cols = [
    'department',
    'region',
    'education',
    'gender',
    'recruitment_channel',
]
test_encoded = pd.get_dummies(
    test_df, columns=cat_cols, drop_first=True, dtype=int
)

X_test_unseen = test_encoded.drop(
    columns=['employee_id'], errors='ignore'
).reindex(columns=X.columns, fill_value=0)


test_predictions = best_rf.predict(X_test_unseen)

sample_sub['is_promoted'] = test_predictions

sample_sub.to_csv('final_promotion_predictions.csv', index=False)

print('Predictions saved successfully!')
print(f'Total rows in output: {len(sample_sub)}')
print('Class Distribution:\n', sample_sub['is_promoted'].value_counts())
sample_sub.head()

Predictions saved successfully!
Total rows in output: 23490
Class Distribution:
 is_promoted
0    21589
1     1901
Name: count, dtype: int64


,employee_id,is_promoted
0,8724,0
1,74430,0
2,72255,0
3,38562,0
4,64486,0
